In [1]:
import pyro
import pyro.distributions as dist
import torch
import numpy as np
import pandas as pd
import plotly as py
import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import ortho_group
import matplotlib.pylab as plt

In [2]:
pyro.set_rng_seed(1337)

In [3]:
N = 1000
D = 3
K = 2

In [4]:
W = pyro.sample('W', dist.Normal(0.,1.).expand([K,D]))

In [5]:
W

tensor([[-2.0260, -2.0655, -1.2054],
        [-0.9122, -1.2502,  0.8032]])

In [6]:
z = pyro.sample('W', dist.Normal(0.,1.).expand([N,K]))

In [7]:
sigma = pyro.sample('sigma', dist.LogNormal(-2.,1.).expand([D]))
#sigma = pyro.sample('sigma', dist.InverseGamma(1.,1.).expand([D]))

In [8]:
Sigma = torch.diag(sigma)

In [9]:
Sigma

tensor([[0.0998, 0.0000, 0.0000],
        [0.0000, 0.0497, 0.0000],
        [0.0000, 0.0000, 0.0358]])

In [10]:
x = pyro.sample('x', dist.MultivariateNormal(torch.matmul(z,W),Sigma))

In [11]:
X = x.T.numpy()

In [12]:
Q = ortho_group.rvs(dim=D)

In [13]:
Wvecs = W.T.numpy()

In [14]:
Wvecs

array([[-2.0259922 , -0.9122241 ],
       [-2.0655088 , -1.2502218 ],
       [-1.2053628 ,  0.80323374]], dtype=float32)

In [15]:
WQ = np.matmul(W,Q)

In [16]:
def cone_obj(vectors):
    return  go.Cone(x=vectors[0],
                   y=vectors[1], 
                   z=vectors[2], 
                   u=vectors[0], 
                   v=vectors[1], 
                   w=vectors[2])

In [17]:
WQ

tensor([[ 3.0855,  0.4918,  0.2484],
        [ 1.0877,  0.7805, -1.1172]], dtype=torch.float64)

In [26]:
Z = z.T.numpy()

In [33]:
data=go.Scatter(
    x=Z[0], y=Z[1], mode='markers',
    marker=dict(size=5),opacity=.6)
fig = go.Figure()
fig.add_trace(data)
#fig.add_trace(Wvectors)
#fig.add_trace(WQvectors)
#fig.add_trace(Rvectors)
fig.update_yaxes(
    scaleanchor = "x",
    scaleratio = 1,
  )
fig.show()

In [32]:
data=go.Scatter3d(
    x=X[0], y=X[1], z=X[2], mode='markers',
    marker=dict(size=2),opacity=.6)
Wvectors = cone_obj(Wvecs)
Rvectors = cone_obj(Q)
WQvectors = cone_obj(WQ.T.numpy())
layout = go.Layout(
             scene=dict(
                 aspectmode='data'
         ))
fig = go.Figure(layout=layout)
fig.add_trace(data)
#fig.add_trace(Wvectors)
#fig.add_trace(WQvectors)
#fig.add_trace(Rvectors)
fig.show()

In [36]:
fig = go.Figure(layout=layout)
fig.add_trace(Wvectors)
fig.show()

In [19]:
py.io.write_html(fig, 'plotly_factor_analysis.html')

In [20]:
py.plot(fig, filename='plotly_factor_analysis')

TypeError: plot() missing 1 required positional argument: 'kind'

In [ ]:
N = 1000
D = 256
K = 2
filter_len = 3

In [ ]:
W = pyro.sample('W', dist.Normal(0.,1.).expand([K,filter_len]))

In [ ]:
z = pyro.sample('W', dist.Laplace(0.,1.).expand([N,K,D]))

In [ ]:
sigma = pyro.sample('sigma', dist.LogNormal(-2.,1.).expand([D]))

In [ ]:
z[torch.abs(z)<1.0] = 0

In [ ]:
%matplotlib qt
plt.imshow(z[:,0,:])

In [ ]:
z.shape

In [ ]:
W.T.unsqueeze(-1).shape

In [ ]:
tmp = torch.conv1d(z, W.T.unsqueeze(-1))

In [ ]:
tmp.sum(1).shape

In [ ]:
Sigma = torch.diag(sigma)

In [ ]:
x = pyro.sample('x', dist.MultivariateNormal(tmp.sum(1),Sigma))

In [ ]:
x.shape

In [ ]:
%matplotlib qt
plt.imshow(x)

In [ ]:
W.shape

In [ ]:
%matplotlib inline

In [ ]:
plt.plot(z[0,0,:])
plt.plot(z[0,1,:])

In [ ]:
plt.plot(x[0,:])

In [ ]:
x[0]